
# Step 1b: Grad‑CAM Inspection Notebook

Use this notebook to **visualize what your classifier is focusing on** (e.g., human vs. avatar vs. animal).  
It supports:
- Loading a SavedModel (`.keras` or SavedModel dir) or a compiled `tf.keras` model
- Directory or CSV dataset indexing
- Generating Grad‑CAM heatmaps for **misclassified** and **correct** examples per class
- Saving side‑by‑side overlays to disk for reports
- (Optional) A simple **border‑attention score** that can hint at shortcut risks (logos/borders)



## 0) Requirements

```bash
pip install tensorflow pillow opencv-python-headless numpy pandas matplotlib tqdm scikit-learn
```



## 1) User Tunables


In [28]:
from pathlib import Path

# ===== USER TUNABLES =====
# MODEL_PATH = Path("models/baseline_savedmodel/resnet50_profilepic_classifier.keras")  # dir or file (.keras / .h5 / SavedModel dir)
MODEL_PATH = Path("models/resnet50_profilepic_classifier.keras")
CLASS_NAMES = None

DATA_ROOT = Path("data//final")
METADATA_CSV = None

TARGET_SPLIT = "val"
IMG_SIZE = 224
BATCH = 32

PREPROCESS = "resnet50"  # 'resnet50' | 'efficientnet' | 'none'
TARGET_LAYER_NAME = None # e.g., "conv5_block3_out"; None => auto-detect last conv
ALPHA = 0.35

N_MISCLASS_PER_CLASS = 8
N_CORRECT_PER_CLASS = 6

OUT_DIR = Path("gradcam_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# =========================


## 2) Imports & GPU Check


In [29]:

import os, json, math, random
import numpy as np
import pandas as pd
import tensorflow as tf
try:
    from tensorflow.keras.applications import resnet50, efficientnet
except Exception:
    print('Using keras model instead of tensorflow.')
    from keras.applications import resnet50, efficientnet  # Keras 3 fallback
from typing import List, Tuple
from tqdm import tqdm

from PIL import Image
import matplotlib.pyplot as plt
import cv2

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TF version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



## 3) Load Model


In [30]:
def load_model_any(path: Path) -> tf.keras.Model:
    """Loads a TensorFlow model from various formats (.keras, .h5, SavedModel dir)."""
    if not path.exists():
        raise FileNotFoundError(f"Model path not found: {path}")

    try:
        # Check if the path is a directory (likely a SavedModel)
        if path.is_dir() and (path / 'saved_model.pb').exists():
            print(f"Loading SavedModel from directory: {path}")
            # Use TFSMLayer for SavedModel format
            # Assuming the default serving signature
            m = tf.keras.layers.TFSMLayer(str(path), call_endpoint='serving_default')
        else:
            print(f"Attempting to load model file: {path}")
            # Try loading as .keras or .h5
            m = tf.keras.models.load_model(path, compile=False)
            try:
                m.compile()
            except Exception:
                pass # Model might be loaded for inference only

        print("Model loaded successfully using TFSMLayer or load_model.")
        return m

    except Exception as e:
        raise RuntimeError(f"Error loading model from {path}: {e}")

print(f"Model Path {MODEL_PATH}")
print("Loading model from:", MODEL_PATH.resolve())
model = load_model_any(MODEL_PATH)
# TFSMLayer does not have a summary method like a standard Keras Model
# You might inspect the layer's inputs/outputs if needed
# model.summary() # Commenting out as TFSMLayer doesn't have summary()

Model Path models/resnet50_profilepic_classifier.keras
Loading model from: /content/models/resnet50_profilepic_classifier.keras
Attempting to load model file: models/resnet50_profilepic_classifier.keras
Model loaded successfully using TFSMLayer or load_model.



## 4) Build Dataset Index (Directory or CSV)


In [31]:
def list_images_directory(root: Path, split: str) -> pd.DataFrame:
    print(f"Checking directory: {root / split}") # Added print statement
    rows = []
    base = root / split
    if not base.exists():
        print(f"Directory does not exist: {base}") # Added print statement
        return pd.DataFrame(columns=["path","label","split"])
    for cls_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for img in cls_dir.rglob("*"):
            if img.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}:
                rows.append({"path": str(img.as_posix()), "label": cls_dir.name, "split": split})
    return pd.DataFrame(rows)

if METADATA_CSV:
    df_all = pd.read_csv(METADATA_CSV)
    need = {"path","label","split"}
    if not need.issubset(set(df_all.columns)):
        raise ValueError(f"CSV must contain columns: {need}")
    df_all["path"] = df_all["path"].astype(str)
else:
    df_train = list_images_directory(DATA_ROOT, "train")
    df_val   = list_images_directory(DATA_ROOT, "val")
    df_test  = list_images_directory(DATA_ROOT, "test")
    df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
    print(f"head: {df_all.head()}")
    print(f"shape:{df_all.shape}")

df = df_all[df_all["split"] == TARGET_SPLIT].copy().reset_index(drop=True)

if df.empty:
    raise RuntimeError(f"Failed to build dataset index for split '{TARGET_SPLIT}'. No data found.")

if CLASS_NAMES is None:
    CLASS_NAMES = sorted(df_all["label"].dropna().unique().tolist())

label_to_index = {c:i for i,c in enumerate(CLASS_NAMES)}
index_to_label = {i:c for c,i in label_to_index.items()}

print("Classes:", CLASS_NAMES)
print("Counts:", df["label"].value_counts())

Checking directory: data/final/train
Checking directory: data/final/val
Checking directory: data/final/test
head:                                            path   label  split
0  data/final/train/animal/a5b067d023ca6582.jpg  animal  train
1  data/final/train/animal/44935642e882041b.jpg  animal  train
2  data/final/train/animal/46c81d173da7e995.jpg  animal  train
3  data/final/train/animal/4494d8932021afa0.jpg  animal  train
4  data/final/train/animal/4512ede202084b7f.jpg  animal  train
shape:(30000, 3)
Classes: ['animal', 'avatar', 'human']
Counts: label
animal    1000
avatar    1000
human     1000
Name: count, dtype: int64



## 5) tf.data Pipeline & Preprocessing


In [32]:
# from tensorflow.keras.applications import resnet50, efficientnet

def preprocess_image(path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    if PREPROCESS.lower() == "resnet50":
        img = resnet50.preprocess_input(img)
    elif PREPROCESS.lower() == "efficientnet":
        img = efficientnet.preprocess_input(img)
    else:
        img = img / 255.0
    return img

def build_ds(paths: List[str], labels: List[int], batch=BATCH, shuffle=False) -> tf.data.Dataset:
    x = tf.constant(paths, dtype=tf.string) # Explicitly cast paths to tf.string
    y = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((x,y))
    if shuffle:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=False)
    ds = ds.map(lambda p,l: (preprocess_image(p), tf.one_hot(l, depth=len(CLASS_NAMES))),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

paths = df["path"].tolist()
labels = [label_to_index[l] for l in df["label"].tolist()]
ds_eval = build_ds(paths, labels, batch=BATCH, shuffle=False)


## 6) Predict & Build a Score Table


In [33]:
probs = []
for xb, yb in ds_eval:
    p = model.predict(xb, verbose=0)
    probs.append(p)
probs = np.vstack(probs)

pred_idx = probs.argmax(axis=1)
pred_lbl = [index_to_label[i] for i in pred_idx]
true_lbl = df["label"].tolist()

conf = probs[np.arange(len(probs)), pred_idx]

score_df = pd.DataFrame({
    "path": paths,
    "true_label": true_lbl,
    "pred_label": pred_lbl,
    "pred_idx": pred_idx,
    "true_idx": [label_to_index[t] for t in true_lbl],
    "confidence": conf
})
score_df["is_correct"] = score_df["true_label"] == score_df["pred_label"]

score_df.head()


,path,true_label,pred_label,pred_idx,true_idx,confidence,is_correct
0,data/final/val/animal/aab0cc4594b50a59.jpg,animal,human,2,0,0.999973,False
1,data/final/val/animal/b50ad59b96963d55.jpg,animal,human,2,0,1.000000,False
2,data/final/val/animal/299e5a9bf3747a7b.jpg,animal,human,2,0,1.000000,False
3,data/final/val/animal/faf19fb7df840432.jpg,animal,human,2,0,0.998135,False
4,data/final/val/animal/0657d60be932783b.jpg,animal,human,2,0,1.000000,False


In [ ]:
print("Checking ds_eval contents:")
count = 0
for images, labels in ds_eval.take(5): # Take up to 5 batches to inspect
    print(f"  Batch {count}:")
    print(f"    Images shape: {images.shape}, dtype: {images.dtype}")
    print(f"    Labels shape: {labels.shape}, dtype: {labels.dtype}")
    count += 1

if count == 0:
    print("  ds_eval is empty. No batches were yielded.")
else:
    print(f"  Iterated through {count} batches.")


## 7) Grad‑CAM Utilities


In [ ]:
def find_last_conv_layer(model: tf.keras.Model) -> str:
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

last_conv_name = TARGET_LAYER_NAME or find_last_conv_layer(model)
last_conv = model.get_layer(last_conv_name)
print("Grad‑CAM target layer:", last_conv_name)

# Ensure model.output is treated as a tensor for grad_model
# This might be necessary if loading from .h5 causes issues with symbolic outputs
model_output_tensor = model.output

grad_model = tf.keras.models.Model(
    [model.inputs], [last_conv.output, model_output_tensor]
)

def make_gradcam_heatmap(img_tensor: tf.Tensor, class_index: int) -> np.ndarray:
    with tf.GradientTape() as tape:
        # Pass the input tensor wrapped in a list to match grad_model's expected input structure
        conv_out, preds = grad_model([img_tensor], training=False)
        # Access the model's final output from the list of predictions
        model_output = preds[1]

        if class_index is None:
            class_index = tf.argmax(model_output[0])

        # Calculate the target score using one-hot encoding and reduce_sum
        target_class_one_hot = tf.one_hot([class_index], depth=tf.shape(model_output)[1])
        target = tf.reduce_sum(model_output * target_class_one_hot, axis=1)


    grads = tape.gradient(target, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(1,2))
    conv_out = conv_out[0]
    pooled_grads = pooled_grads[0]

    heatmap = tf.tensordot(conv_out, pooled_grads, axes=(2,0))
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0, 0)
    overlay = overlay[:, :, ::-1]
    return img_np, overlay

def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b>0 and (h-2*b)>0 and (w-2*b)>0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)

In [ ]:
# Build grad_model exactly from the original model's inputs/outputs
last_conv_name = TARGET_LAYER_NAME or find_last_conv_layer(model)
last_conv = model.get_layer(last_conv_name)
grad_model = tf.keras.Model(inputs=model.inputs, outputs=[last_conv.output, model.output])

def _call_with_structured_inputs(m, x):
    """
    Try calling model with bare tensor, [tensor], and {input_name: tensor}.
    Works around Keras 3 structured-input expectations (e.g., Input(name="image", ...)).
    """
    # 1) Try bare tensor
    try:
        return m(x, training=False)
    except Exception:
        pass
    # 2) Try list-wrapped (single-input models often accept [x])
    try:
        return m([x], training=False)
    except Exception:
        pass
    # 3) Try dict by first input name
    try:
        names = getattr(m, "input_names", None)
        if names and len(names) == 1:
            return m({names[0]: x}, training=False)
    except Exception:
        pass
    # If still failing, raise a clear error
    raise RuntimeError(
        f"Could not call model with structured inputs. "
        f"Input names={getattr(m, 'input_names', None)}; got tensor shape={x.shape}"
    )

def make_gradcam_heatmap(img_tensor: tf.Tensor, class_index: int) -> np.ndarray:
    with tf.GradientTape() as tape:
        conv_out, preds = _call_with_structured_inputs(grad_model, img_tensor)
        if class_index is None:
            class_index = int(tf.argmax(preds[0]))
        target = preds[:, class_index]

    grads = tape.gradient(target, conv_out)                  # (1, h, w, c)
    pooled_grads = tf.reduce_mean(grads, axis=(1, 2))        # (1, c)
    conv_out = conv_out[0]                                   # (h, w, c)
    pooled_grads = pooled_grads[0]                           # (c,)
    heatmap = tf.tensordot(conv_out, pooled_grads, axes=(2, 0))
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# Also update single-image helper to predict robustly:
def _predict_structured(m, x):
    try:
        return m.predict(x, verbose=0)
    except Exception:
        try:
            return m.predict([x], verbose=0)
        except Exception:
            names = getattr(m, "input_names", None)
            if names and len(names) == 1:
                return m.predict({names[0]: x}, verbose=0)
            raise

def gradcam_single_image(path: str, class_idx: int = None):
    x = preprocess_for_single(path)
    if class_idx is None:
        p = _predict_structured(model, x)[0]
        class_idx = int(np.argmax(p))
    heat = make_gradcam_heatmap(x, class_index=class_idx)
    img_np, overlay = overlay_heatmap_on_image(path, heat)
    frac = border_attention_fraction(heat)
    # ... (plot as before)



## 8) Visualize Grad‑CAM Panels


In [ ]:

# import math

# def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
#     rows = []
#     for cls in CLASS_NAMES:
#         sub = score_df[score_df["true_label"] == cls].copy()
#         mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
#         ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
#         rows.append(("MIS", cls, mis))
#         rows.append(("OK",  cls, ok))
#     return rows

def preprocess_for_single(path: str) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    if PREPROCESS.lower() == "resnet50":
        from tensorflow.keras.applications.resnet50 import preprocess_input
        img = preprocess_input(img)
    elif PREPROCESS.lower() == "efficientnet":
        from tensorflow.keras.applications.efficientnet import preprocess_input
        img = preprocess_input(img)
    else:
        img = img / 255.0
    return tf.expand_dims(img, 0)

# def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
#     n = len(group_df)
#     if n == 0:
#         return

#     cols = 3
#     rows = n
#     fig_h = max(4, rows * 3)
#     fig_w = 12

#     plt.figure(figsize=(fig_w, fig_h))
#     idx = 1
#     records = []

#     for r in group_df.itertuples(index=False):
#         x = preprocess_for_single(r.path)
#         heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
#         img_np, overlay = overlay_heatmap_on_image(r.path, heat)
#         frac = border_attention_fraction(heat)

#         plt.subplot(n, cols, idx);   plt.imshow(img_np);  plt.axis("off");
#         plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
#         idx += 1
#         plt.subplot(n, cols, idx);   plt.imshow(heat, cmap="jet");  plt.axis("off");  plt.title("Heatmap")
#         idx += 1
#         plt.subplot(n, cols, idx);   plt.imshow(overlay); plt.axis("off");
#         plt.title(f"Overlay\nborder={frac:.2f}")
#         idx += 1

#         records.append({
#             "path": r.path,
#             "true_label": r.true_label,
#             "pred_label": r.pred_label,
#             "confidence": r.confidence,
#             "border_attention_frac": frac,
#             "is_correct": r.is_correct
#         })

#     plt.suptitle(f"{kind}: {cls} — {n} samples", y=1.02)
#     plt.tight_layout()
#     plt.savefig(save_path, dpi=160, bbox_inches="tight")
#     plt.show()

#     pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)

# samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)
# for kind, cls, gdf in samples:
#     slug = f"{kind.lower()}_{cls}".replace(" ", "_")
#     out_file = OUT_DIR / f"gradcam_{slug}.png"
#     panel_for_group(kind, cls, gdf, out_file)

# print("Saved panels to:", OUT_DIR.resolve())



## 9) Single‑Image Helper


In [ ]:

def gradcam_single_image(path: str, class_idx: int = None):
    x = preprocess_for_single(path)
    if class_idx is None:
        p = model.predict(x, verbose=0)[0]
        class_idx = int(np.argmax(p))
    heat = make_gradcam_heatmap(x, class_index=class_idx)
    img_np, overlay = overlay_heatmap_on_image(path, heat)
    frac = border_attention_fraction(heat)

    plt.figure(figsize=(10,3))
    plt.subplot(1,3,1); plt.imshow(img_np); plt.axis("off"); plt.title("Original")
    plt.subplot(1,3,2); plt.imshow(heat, cmap="jet"); plt.axis("off"); plt.title("Heatmap")
    plt.subplot(1,3,3); plt.imshow(overlay); plt.axis("off"); plt.title(f"Overlay\nborder={frac:.2f}")
    plt.tight_layout()
    plt.show()

# Example:
# gradcam_single_image(df.iloc[0]['path'])



## 10) What to Look For

- **Correct**: heat concentrated on salient object (face/body/animal).
- **Wrong**: heat on **backgrounds, borders, logos, corner watermarks**, or text.
- If border‑attention fractions are systematically high, consider **masking/cropping** or **augmentations** that randomize edges.


# Tools

In [ ]:
# Install the Google Cloud Storage client library
# !pip install google-cloud-storage

Authenticating with the service account key and downloading files from the bucket.

Using `gsutil` with multiple threads for faster downloads.

## Load Data (Pictures) from GCS

In [ ]:
def load_files():
  # Specify the bucket name and the local directory to save the files
  bucket_name = 'gs://along-capstone-data'
  source_directory = 'final'
  destination_directory = 'data'

  # Create the destination directory if it doesn't exist
  import os
  os.makedirs(destination_directory, exist_ok=True)

  import json
  from google.colab import userdata
  import time

  # Get the service account key from Colab Secrets
  service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

  # Define the path to save the service account key file
  key_file_path = 'service_account_key.json'

  # Save the service account key to a file
  with open(key_file_path, 'w') as f:
      json.dump(service_account_info, f)

  # Authenticate gcloud and gsutil using the service account key file
  !gcloud auth activate-service-account --key-file {key_file_path}

  start_time = time.time()

  # Download files using gsutil with multiple threads
  # -m enables multithreading
  # -r recursively copies directories and files
  !gsutil -m cp -r {bucket_name}/{source_directory} {destination_directory}

  end_time = time.time()
  elapsed_time = end_time - start_time

  print("Download complete.")
  print(f"Load operation took {elapsed_time:.2f} seconds.")


load_files()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# List the contents of the specified directory in Google Drive
# !ls /content/drive/MyDrive/"Colab Data"/Capstone

In [ ]:
def load_files():
  import os
  from pathlib import Path

  # Define source and destination paths
  source_path = Path('/content/drive/MyDrive/Colab Data/Capstone/resnet50_profilepic_classifier.keras')
  destination_dir = Path('models/baseline_savedmodel')
  destination_path = destination_dir / source_path.name

  # Create the destination directory if it doesn't exist
  destination_dir.mkdir(parents=True, exist_ok=True)

  # Copy the file using shell command
  !cp "{source_path}" "{destination_path}"

  print(f"Copied {source_path} to {destination_path}")

# load_files()

## Load Model from GCS

In [ ]:
def load_model_from_gcs(bucket_name='gs://along-capstone-data', model_name="resnet50_profilepic_classifier.keras", directory='models'):
  """Saves a Keras model to a Google Cloud Storage bucket."""
  import json
  from google.colab import userdata
  from pathlib import Path
  import os

  # Get the service account key from Colab Secrets
  service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

  # Define the path to save the service account key file
  key_file_path = 'service_account_key.json'

  # Save the service account key to a file
  with open(key_file_path, 'w') as f:
      json.dump(service_account_info, f)

  # Authenticate gcloud and gsutil using the service account key file
  !gcloud auth activate-service-account --key-file {key_file_path}

  # Define the local path to save the model temporarily
  local_model_dir = Path(directory)
  local_model_dir.mkdir(parents=True, exist_ok=True)
  # local_model_path = local_model_dir / model_name

  # Load the model locally
  # model_to_save.save(local_model_path)
  # print(f"Model saved locally to {local_model_path}")

  # Upload the model to GCS
  gcs_model_path = f"{bucket_name}/{directory}/{model_name}"
  !gsutil cp {gcs_model_path} {local_model_dir}
  print(f"Model loaded to {gcs_model_path}")

  # # Clean up the local model file and directory
  # local_model_path.unlink()
  # local_model_dir.rmdir() # This will only work if the directory is empty after deleting the model file
  # print(f"Local model file {local_model_path} and directory {local_model_dir} removed.")

# Example call (uncomment to use):
load_model_from_gcs()